# Fine-tune DeepSeek-R1 Distill Llama 8B with Unsloth + LoRA

This notebook fine-tunes a DeepSeek-R1-Distill-Llama 8B model using Unsloth + LoRA on a medical reasoning SFT dataset.

**Key Features:**
- Two training modes: quick test (~4-5 min) or production (~2-4 hours)
- Pre and post-training inference checks
- Parameterized training configuration
- Auto-saves fine-tuned model

**Assumptions:**
- CUDA-capable GPU available to PyTorch
- Model at: `./local-DeepSeek-R1-Distill-Llama-8B`
- Dataset at: `./medical-o1-reasoning-SFT/hf_format`

## Configuration: Select Training Mode

Set `MODE` to choose between quick testing or production training:

In [64]:
# Choose training mode: "test" or "production"
MODE = "test"  # Change to "production" for full training

# Mode summary
print("\\n" + "="*70)
print(f"  FINE-TUNING MODE: {MODE.upper()}")
print("="*70)
if MODE == "test":
    print("  Quick Test Run: 10 steps, ~4-5 minutes")
    print("  Use for: Pipeline validation and debugging")
else:
    print("  Production Training: 3 epochs, ~2-4 hours")
    print("  Use for: Full model fine-tuning for deployment")
print("="*70 + "\\n")

\n======================================================================
  FINE-TUNING MODE: TEST
  Quick Test Run: 10 steps, ~4-5 minutes
  Use for: Pipeline validation and debugging
======================================================================\n


## Step 1: Import Libraries

In [65]:
from unsloth import FastLanguageModel
import torch
#import os
from datasets import load_from_disk
from trl import SFTTrainer, SFTConfig
#from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

print("✅ All imports successful")

✅ All imports successful


## Step 2: Check CUDA Availability

In [66]:
if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is not available to PyTorch. "
        "Fix your PyTorch + CUDA install before running fine-tuning."
    )

print(f"✅ CUDA available: {torch.cuda.is_available()}")
print(f"   GPU: {torch.cuda.get_device_name(0)}")
print(f"   Max memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

✅ CUDA available: True
   GPU: NVIDIA GB10
   Max memory: 121.7 GB


## Step 3: Load Model and Tokenizer

In [67]:
max_seq_length = 2048
dtype = None
load_in_4bit = True

print("Loading model with 4-bit quantization...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="./local-DeepSeek-R1-Distill-Llama-8B",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)
print("✅ Model loaded successfully")

Loading model with 4-bit quantization...
==((====))==  Unsloth 2026.7.6: Fast Llama patching. Transformers: 5.14.1.
   \\   /|    NVIDIA GB10. Num GPUs = 1. Max memory: 121.693 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 12.1. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load ./local-DeepSeek-R1-Distill-Llama-8B as a legacy tokenizer.


✅ Model loaded successfully


## Step 4: Define Prompt Templates

In [68]:
prompt_style = """Below is an instruction that describes a task, paired with an input that provides further context.
Write a response that appropriately completes the request.
Before answering, think carefully about the question and create a step-by-step chain of thoughts to ensure a logical and accurate response.

### Instruction:
You are a medical expert with advanced knowledge in clinical reasoning, diagnostics, and treatment planning.
Please answer the following medical question.

### Question:
{}

### Response:
<think>{}"""

train_prompt_style = """Below is an instruction that describes a task, paired with an input that provides further context.
Write a response that appropriately completes the request.
Before answering, think carefully about the question and create a step-by-step chain of thoughts to ensure a logical and accurate response.

### Instruction:
You are a medical expert with advanced knowledge in clinical reasoning, diagnostics, and treatment planning.
Please answer the following medical question.

### Question:
{}

### Response:
<think>
{}
</think>
{}"""

print("✅ Prompt templates defined")

✅ Prompt templates defined


## Step 5: Pre-Training Inference Check

In [69]:
question = (
    "A 69-year-old man is experiencing burning pain, tingling, numbness, itching, "
    "and pins-and-needles sensations over the outer right thigh after 15-20 minutes "
    "of standing. The symptoms go away after sitting down. The man has diabetes, "
    "but controlled. What could be the possible cause(s) of his symptoms?"
)

FastLanguageModel.for_inference(model)
inputs = tokenizer([prompt_style.format(question, "")], return_tensors="pt").to("cuda")

outputs = model.generate(
    input_ids=inputs.input_ids,
    attention_mask=inputs.attention_mask,
    max_new_tokens=1200,
    use_cache=True,
)

response = tokenizer.batch_decode(outputs, skip_special_tokens=True, clean_up_tokenization_spaces=False)
response_text = response[0]

# Fix BPE space token artifacts FIRST (Ġ → space, Ċ → newline)
response_text = response_text.replace("Ġ", " ").replace("Ċ", "\n").replace("  ", " ")

print("\n" + "="*70)
print("PRE-TRAINING INFERENCE")
print("="*70 + "\n")

# Extract from ### Response: onward
if "### Response:" in response_text:
    response_only = response_text.split("### Response:")[1].strip()
    print(response_only)
elif "<think>" in response_text:
    # Fallback: extract from <think> onward
    response_only = response_text.split("<think>")[1].strip()
    print("<think>" + response_only)
else:
    print(response_text.strip())

Both `max_new_tokens` (=1200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



PRE-TRAINING INFERENCE

<think>Okay, so I'm trying to figure out why a 69-year-old man with controlled diabetes is experiencing burning pain, tingling, numbness, itching, and pins-and-needles sensations over the outer right thigh after standing for 15-20 minutes. The symptoms go away when he sits down. Hmm, let's break this down.

First, the key points: male, 69 years old, diabetes controlled, symptoms after standing, symptoms resolve when sitting. So, the issue seems to be related to standing. That makes me think about things that happen when you're upright. Maybe something to do with blood flow or nerve compression.

The symptoms include burning pain, tingling, numbness, itching, and pins-and-needles. These are all sensations that suggest some kind of nerve dysfunction. Pins-and-needles usually indicate a temporary, possibly reversible nerve injury, like something that happens when there's pressure on a nerve.

The fact that it happens after standing makes me think about positional 

## Step 6: Load and Format Dataset

In [70]:
EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
    inputs = examples["Question"]
    cots = examples["Complex_CoT"]
    outputs = examples["Response"]

    texts = []
    for q, cot, ans in zip(inputs, cots, outputs):
        text = train_prompt_style.format(q, cot, ans) + EOS_TOKEN
        texts.append(text)

    return {"text": texts}

print("Loading dataset from disk...")
dataset_path = "./medical-o1-reasoning-SFT/hf_format"
dataset_on_disk = load_from_disk(dataset_path, "en")

if hasattr(dataset_on_disk, "keys"):
    print(f"Detected DatasetDict splits: {list(dataset_on_disk.keys())}")
    base_train = dataset_on_disk["train"]
else:
    base_train = dataset_on_disk

base_train = base_train.shuffle(seed=42)
N = min(16000, len(base_train))
train_dataset = base_train.select(range(N))

print(f"Rows in raw training subset: {len(train_dataset)}")

# Display sample BEFORE formatting
print("\n" + "="*70)
print("BEFORE FORMATTING (Raw Data from Disk)")
print("="*70)
sample_question = train_dataset["Question"][0]
sample_cot = train_dataset["Complex_CoT"][0]
sample_response = train_dataset["Response"][0]

print(f"\n📝 Question:\n{sample_question[:400]}{'...' if len(sample_question) > 400 else ''}")
print(f"\n💭 Complex Chain of Thought (first 400 chars):\n{sample_cot[:400]}{'...' if len(sample_cot) > 400 else ''}")
print(f"\n📋 Response (first 400 chars):\n{sample_response[:400]}{'...' if len(sample_response) > 400 else ''}")

dataset = train_dataset.map(formatting_prompts_func, batched=True)
print(f"\nRows after formatting: {len(dataset)}")

# Display same sample AFTER formatting
print("\n" + "="*70)
print("AFTER FORMATTING (Complete Training Example)")
print("="*70)
print(f"\n✨ Formatted text (full structure with prompt template and EOS token):\n")
print(dataset["text"][0])

Loading dataset from disk...
Detected DatasetDict splits: ['train']
Rows in raw training subset: 16000

BEFORE FORMATTING (Raw Data from Disk)

📝 Question:
In the instrument formula for a Gingival Margin Trimmer (GMT) used during cavity preparation, what is the second number representing the angle of the cutting edge when access to the distal gingival margin is achieved?

💭 Complex Chain of Thought (first 400 chars):
Alright, so a Gingival Margin Trimmer, or GMT for short, is some sort of dental tool used during cavity prep. I need to figure out what that second number in its formula really means, especially when working with the distal gingival margin. Let's start with the basics about these numbers.

The first number in any dental instrument formula usually tells us the blade width, measured in tenths of a m...

📋 Response (first 400 chars):
In the instrument formula for a Gingival Margin Trimmer (GMT) used during cavity preparation, the second number, which represents the angle of t

Map:   0%|          | 0/16000 [00:00<?, ? examples/s]


Rows after formatting: 16000

AFTER FORMATTING (Complete Training Example)

✨ Formatted text (full structure with prompt template and EOS token):

Below is an instruction that describes a task, paired with an input that provides further context.
Write a response that appropriately completes the request.
Before answering, think carefully about the question and create a step-by-step chain of thoughts to ensure a logical and accurate response.

### Instruction:
You are a medical expert with advanced knowledge in clinical reasoning, diagnostics, and treatment planning.
Please answer the following medical question.

### Question:
In the instrument formula for a Gingival Margin Trimmer (GMT) used during cavity preparation, what is the second number representing the angle of the cutting edge when access to the distal gingival margin is achieved?

### Response:
<think>
Alright, so a Gingival Margin Trimmer, or GMT for short, is some sort of dental tool used during cavity prep. I need to figur

## Step 7: Apply LoRA Adapters

In [ ]:
print("Applying LoRA adapters...")
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)
print("✅ LoRA adapters applied")

## Step 8: Configure Trainer

In [ ]:
print(f"Configuring trainer for {MODE} mode...")

# Build trainer kwargs
trainer_kwargs = {
    "model": model,
    "tokenizer": tokenizer,
    "train_dataset": dataset,
    "dataset_text_field": "text",
    "max_seq_length": max_seq_length,
    "packing": True,  # Enable packing for Unsloth's padding_free optimization
}

if MODE == "test":
    trainer = SFTTrainer(
        **trainer_kwargs,
        args=SFTConfig(
            per_device_train_batch_size=2,
            gradient_accumulation_steps=4,
            warmup_steps=5,
            max_steps=10,
            learning_rate=2e-4,
            fp16=not is_bfloat16_supported(),
            bf16=is_bfloat16_supported(),
            logging_steps=10,
            optim="adamw_8bit",
            weight_decay=0.01,
            lr_scheduler_type="linear",
            seed=3407,
            output_dir="outputs",
            save_strategy="no",
            packing=True,  # Explicitly set packing in SFTConfig
            packing_strategy="bfd",  # Block-Fit Dot product strategy
        ),
    )
    print("✅ Loaded TEST configuration (10 steps, ~4-5 minutes)\n")
else:
    trainer = SFTTrainer(
        **trainer_kwargs,
        args=SFTConfig(
            per_device_train_batch_size=4,
            gradient_accumulation_steps=8,
            num_train_epochs=3,
            warmup_ratio=0.03,
            learning_rate=1e-4,
            fp16=not is_bfloat16_supported(),
            bf16=is_bfloat16_supported(),
            logging_steps=25,
            optim="adamw_8bit",
            weight_decay=0.01,
            lr_scheduler_type="cosine",
            seed=3407,
            output_dir="outputs",
            save_strategy="no",
            packing=True,  # Explicitly set packing in SFTConfig
            packing_strategy="bfd",  # Block-Fit Dot product strategy
        ),
    )
    print("✅ Loaded PRODUCTION configuration (3 epochs, ~2-4 hours)\n")

## Step 9: Train the Model

In [ ]:
print("\\n" + "="*70)
print("  TRAINING IN PROGRESS...")
print("="*70 + "\\n")

trainer_stats = trainer.train()

print("\\n" + "="*70)
print("  TRAINING COMPLETE")
print("="*70 + "\\n")

## Step 10: Post-Training Inference Check

In [ ]:
FastLanguageModel.for_inference(model)
inputs = tokenizer([prompt_style.format(question, "")], return_tensors="pt").to("cuda")

outputs = model.generate(
    input_ids=inputs.input_ids,
    attention_mask=inputs.attention_mask,
    max_new_tokens=1200,
    use_cache=True,
)

response = tokenizer.batch_decode(outputs, skip_special_tokens=True, clean_up_tokenization_spaces=False)
response_text = response[0]

# Fix BPE space token artifacts FIRST (Ġ → space, Ċ → newline)
response_text = response_text.replace("Ġ", " ").replace("Ċ", "\n").replace("  ", " ")

print("\n" + "="*70)
print("POST-TRAINING INFERENCE")
print("="*70 + "\n")

# Extract from ### Response: onward
if "### Response:" in response_text:
    response_only = response_text.split("### Response:")[1].strip()
    print(response_only)
elif "<think>" in response_text:
    # Fallback: extract from <think> onward
    response_only = response_text.split("<think>")[1].strip()
    print("<think>" + response_only)
else:
    print(response_text.strip())

## Step 11: Save Fine-Tuned Model

In [ ]:
new_model_local = "DeepSeek-R1-Medical-FT-8b-16bts"

print(f"Saving LoRA adapters to: {new_model_local}")
model.save_pretrained(new_model_local)
tokenizer.save_pretrained(new_model_local)

print(f"Saving merged model to: {new_model_local}")
model.save_pretrained_merged(
    new_model_local,
    tokenizer,
    save_method="merged_16bit",
)

print(f"\\n✅ Fine-tuned model saved to: {new_model_local}")